## Libraries

In [61]:
import boto3, textwrap
import json
BUCKET = "coachai-mvp-media"
KEY = "IMG_8831.mov"   # can be .mov/.mp4/.mkv OR audio
CHUNK_SECONDS = 900    # 15 minutes


### Video or Audio Upload to S3 Bucket Transcription

In [9]:
s3 = boto3.client("s3")

BUCKET = "coachai-mvp-media"
SCRIPT_KEY = "code/process_media.py"   # ✅ script location

script = r'''
import argparse, os, subprocess, glob, boto3, whisper

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--bucket", required=True)
    p.add_argument("--input_key", required=True)
    p.add_argument("--out_prefix", required=True)
    p.add_argument("--chunk_seconds", type=int, default=900)
    p.add_argument("--model_size", default="base")
    args = p.parse_args()

    s3 = boto3.client("s3")

    os.makedirs("/opt/ml/processing/tmp", exist_ok=True)
    local_media = "/opt/ml/processing/tmp/input"
    local_wav = "/opt/ml/processing/tmp/audio.wav"
    chunks_dir = "/opt/ml/processing/tmp/chunks"
    os.makedirs(chunks_dir, exist_ok=True)

    # Download input media from S3
    s3.download_file(args.bucket, args.input_key, local_media)

    # Extract audio to WAV 16k mono (ffmpeg exists in processing container)
    subprocess.run([
        "ffmpeg", "-y", "-i", local_media,
        "-vn", "-ac", "1", "-ar", "16000",
        local_wav
    ], check=True)

    # Chunk WAV into segments
    subprocess.run([
        "ffmpeg", "-i", local_wav,
        "-f", "segment",
        "-segment_time", str(args.chunk_seconds),
        "-reset_timestamps", "1",
        f"{chunks_dir}/chunk_%03d.wav"
    ], check=True)

    model = whisper.load_model(args.model_size)

    parts = []
    for wav in sorted(glob.glob(f"{chunks_dir}/*.wav")):
        r = model.transcribe(wav, language="nl", fp16=False)
        parts.append((r.get("text") or "").strip())

    transcript = "\\n\\n".join([p for p in parts if p]).strip()

    s3.put_object(
        Bucket=args.bucket,
        Key=f"{args.out_prefix}/whisper.txt",
        Body=transcript.encode("utf-8"),
        ContentType="text/plain; charset=utf-8"
    )

    print("DONE, wrote transcript to:", f"{args.out_prefix}/whisper.txt")

if __name__ == "__main__":
    main()
'''

s3.put_object(
    Bucket=BUCKET,
    Key=SCRIPT_KEY,
    Body=textwrap.dedent(script).encode("utf-8"),
    ContentType="text/x-python; charset=utf-8"
)

print("✅ Uploaded script to:", f"s3://{BUCKET}/{SCRIPT_KEY}")


✅ Uploaded script to: s3://coachai-mvp-media/code/process_media.py


In [11]:
import boto3
s3 = boto3.client("s3")

h = s3.head_object(Bucket="coachai-mvp-media", Key="IMG_8831.mov")
print("bytes:", h["ContentLength"])
print("GB:", h["ContentLength"]/1024/1024/1024)


bytes: 4168673767
GB: 3.8823799854144454


## Run.sh Setup

In [39]:
import boto3

BUCKET="coachai-mvp-media"
RUNSH_KEY="code/run.sh"

run_sh = """#!/usr/bin/env bash
set -euo pipefail

echo "Installing ffmpeg..."
apt-get update -y
apt-get install -y ffmpeg

echo "Upgrading pip..."
python -m pip install -q --upgrade pip

echo "Installing pinned deps (fix numpy/numba mismatch)..."
pip install -q --no-cache-dir \
  "numpy==1.26.4" \
  "llvmlite==0.42.0" \
  "numba==0.59.1"

echo "Installing torch CPU..."
pip install -q --no-cache-dir \
  "torch==2.1.2" \
  "torchaudio==2.1.2" \
  --index-url https://download.pytorch.org/whl/cpu

echo "Installing Whisper..."
pip install -q --no-cache-dir "openai-whisper==20231117"

echo "Running processing script..."
python /opt/ml/processing/code/process_media.py \\
  --bucket "$BUCKET" \\
  --input_key "$VIDEO_KEY" \\
  --out_prefix "$OUT_PREFIX" \\
  --chunk_seconds "$CHUNK_SECONDS" \\
  --model_size "$MODEL_SIZE"

echo "DONE"
"""

boto3.client("s3").put_object(
    Bucket=BUCKET,
    Key=RUNSH_KEY,
    Body=run_sh.encode("utf-8"),
    ContentType="text/x-shellscript; charset=utf-8"
)

print("✅ Updated run.sh:", f"s3://{BUCKET}/{RUNSH_KEY}")


✅ Updated run.sh: s3://coachai-mvp-media/code/run.sh


### Image URI

In [40]:
import boto3

region = boto3.Session().region_name
print("Region:", region)

# SageMaker built-in image retrieval
from sagemaker.image_uris import retrieve

image_uri = retrieve(
    framework="sklearn",
    region=region,
    version="1.2-1",
    instance_type="ml.m5.large",   # just for selection logic
    image_scope="training"
)

print("Image URI:", image_uri)


Region: eu-central-1
Image URI: 492215442770.dkr.ecr.eu-central-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3


### Transcription

In [41]:
import boto3, uuid

BUCKET = "coachai-mvp-media"
VIDEO_KEY = "IMG_8831.mov"
OUT_PREFIX = "outputs/IMG_8831"

ROLE_ARN = "arn:aws:iam::646011843687:role/service-role/AmazonSageMaker-ExecutionRole-20260118T183678"

REGION = boto3.Session().region_name
from sagemaker.image_uris import retrieve
IMAGE_URI = retrieve(
    framework="sklearn",
    region=REGION,
    version="1.2-1",
    instance_type="ml.m5.large",
    image_scope="training"
)

sm = boto3.client("sagemaker")
job_name = f"coachai-proc-{uuid.uuid4().hex[:8]}"

sm.create_processing_job(
    ProcessingJobName=job_name,
    RoleArn=ROLE_ARN,
    AppSpecification={
        "ImageUri": IMAGE_URI,
        "ContainerEntrypoint": ["bash", "/opt/ml/processing/code/run.sh"]
    },
    ProcessingInputs=[
        {
            "InputName": "code",
            "S3Input": {
                "S3Uri": f"s3://{BUCKET}/code/",
                "LocalPath": "/opt/ml/processing/code",
                "S3DataType": "S3Prefix",
                "S3InputMode": "File",
                "S3DataDistributionType": "FullyReplicated"
            }
        }
    ],
    Environment={
        "BUCKET": BUCKET,
        "VIDEO_KEY": VIDEO_KEY,
        "OUT_PREFIX": OUT_PREFIX,
        "CHUNK_SECONDS": "900",
        "MODEL_SIZE": "base"
    },
    ProcessingResources={
        "ClusterConfig": {
            "InstanceCount": 1,
            "InstanceType": "ml.t3.medium",   # keep what works for your quota
            "VolumeSizeInGB": 300
        }
    },
    StoppingCondition={"MaxRuntimeInSeconds": 8*60*60}
)

print("✅ Started processing job:", job_name)
print("Transcript will be written to:", f"s3://{BUCKET}/{OUT_PREFIX}/whisper.txt")


✅ Started processing job: coachai-proc-a6acd07f
Transcript will be written to: s3://coachai-mvp-media/outputs/IMG_8831/whisper.txt


In [45]:
import boto3

sm = boto3.client("sagemaker")
resp = sm.describe_processing_job(ProcessingJobName="coachai-proc-a6acd07f")

print("Status:", resp["ProcessingJobStatus"])


Status: Completed


In [ ]:
import boto3

BUCKET = "coachai-mvp-media"
KEY = "outputs/IMG_8831/whisper.txt"

s3 = boto3.client("s3")
obj = s3.get_object(Bucket=BUCKET, Key=KEY)
text = obj["Body"].read().decode("utf-8", errors="replace")

print("Characters:", len(text))
print("Words:", len(text.split()))
print("\n--- FIRST 500 CHARS ---\n", text[:500])
print("\n--- LAST 500 CHARS ---\n", text[-500:])


### EMCC Report

In [57]:
import boto3

REGION = boto3.Session().region_name
s3 = boto3.client("s3", region_name=REGION)

BUCKET = "coachai-mvp-media"
TRANSCRIPT_KEY = "outputs/IMG_8831/whisper.txt"

obj = s3.get_object(Bucket=BUCKET, Key=TRANSCRIPT_KEY)
dutch_transcript = obj["Body"].read().decode("utf-8", errors="replace")

print("Transcript length:", len(dutch_transcript))


Transcript length: 61793


In [59]:
EMCC_RULES = """
[PASTE the relevant EMCC / MCC competence descriptions here.
For example:
- Establishing trust and intimacy
- Active listening
- Powerful questioning
- Facilitating awareness
- Managing progress and accountability
etc.]
"""

REPORT_SAMPLE = """
[PASTE a shortened version of the CoachBetter-style report you shared.
Include headings and tone, not client data.]
1. Assessment Summary
This session highlights a strong ability to guide a client toward clear, actionable outcomes. The coach excelled at creating a
structured conversation that moved from identifying a problem to preparing for a solution, demonstrating a supportive and
goal-oriented approach. A key strength was the use of practical tools, like role-playing, to build the client's confidence in
handling a challenging situation. The primary opportunity for growth lies in shifting from a problem-solving focus to a more
holistic, client-centered exploration. By deepening the partnership and inquiring more into the client's internal experience—
their thoughts, feelings, and underlying values—the coach can unlock more profound awareness and learning that extends
beyond the immediate issue.
Competency
1. Demonstrates Ethical Practice
        Level
        No Violation Detected
        2. Embodies a Coaching Mindset
        3. Establishes and Maintains Agreements
        4. Cultivates Trust and Safety
        5. Maintains Presence
        6. Listens Actively
        7. Evokes Awareness
        N/A
        ACC
        ACC
        ACC
        PCC
        ACC
        8. Facilitates Client Growth
2. Demonstrates Ethical Practice:
Summary of Performance: No Violation Detected

3. 3. Establishes and Maintains Agreements: ACC
Summary of Performance:
Suggestions for Advancing to the Next Level:
i. Explore Measures of Success:

ii. Explore Underlying Challenges:

4. Cultivates Trust and Safety: ACC
Summary of Performance:

Suggestions for Advancing to the Next Level:
i. Acknowledge Client Courage
ii. Invite the Client to Co-Create the Process:

5. Maintains Presence: ACC
Summary of Performance:

Suggestions for Advancing to the Next Level:
i. Enhance Partnership in Method Selection

ii. Deepen Curiosity About Client's Perspective:

6. Listens Actively: PCC
Summary of Performance:

Suggestions for Advancing to the Next Level:
i. Explore the Depth of Client Emotions:
ii. Reflect the Client's Whole-Person Context:

7. Evokes Awareness: ACC
Summary of Performance:

Suggestions for Advancing to the Next Level:
i. Ask More Open-Ended, Non-Leading Questions:
ii. Challenge the Client to Explore Beyond Current Thinking:

8. Facilitates Client Growth: ACC
Summary of Performance:
Suggestions for Advancing to the Next Level:
i. Explore Broader Learning from the Situation:
ii. Encourage Client-Led Action Design:

Key Opportunities for Improvement:
Deepen Client Partnership: Shift from suggesting methods like role-playing to co-creating the process with the client. Asking
"What would be most helpful for you right now?" invites the client to be an equal partner in their own journey.
Explore the "Who," Not Just the "What": Move beyond the details of the problem to explore the client's internal
experience. Inquire more deeply into their feelings, values, and beliefs to help them understand themselves better within the
situation.
Connect Actions to Broader Growth: Frame action steps not just as solutions to a problem, but as opportunities for
learning. Ask what the client will learn about themselves by taking a certain action to foster more sustainable growth.
With continued practice, integrating these deeper levels of inquiry will elevate the coaching from effective problem-solving to
transformative personal development.
"""


In [62]:
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

MODEL_ID = "anthropic.claude-3-5-sonnet-20240620-v1:0"

SYSTEM_PROMPT = """
You are a senior EMCC-accredited coaching supervisor and assessor.

Your task:
- Evaluate a coaching conversation transcript
- Assess the coach's performance against EMCC / MCC competencies
- Produce a professional written coaching report

Rules:
- Base judgments ONLY on the transcript
- Do not invent behaviours
- Be balanced, fair, and evidence-based
- Use professional coaching language
- Follow the structure and tone of the provided sample report
"""

USER_PROMPT = f"""
EMCC COMPETENCE FRAMEWORK:
{EMCC_RULES}

SAMPLE REPORT STRUCTURE & STYLE:
{REPORT_SAMPLE}

COACHING TRANSCRIPT (Dutch):
{dutch_transcript}

TASK:
Create a professional coaching evaluation report with the following sections:

1. Session overview
2. Coaching context and contract
3. Observed coaching behaviours
4. EMCC competence alignment (with examples)
5. Strengths demonstrated by the coach
6. Areas for development
7. Overall professional judgment (aligned with EMCC standards)

Write the report in clear, professional English.
"""

body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 4000,
    "temperature": 0.2,
    "system": SYSTEM_PROMPT,
    "messages": [
        {"role": "user", "content": USER_PROMPT}
    ]
}

response = bedrock.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(body)
)

report = json.loads(response["body"].read())["content"][0]["text"]
print(report[:1000])


Here is a professional coaching evaluation report based on the provided transcript:

Coaching Evaluation Report

1. Session Overview

This was the third and final session in a series of coaching conversations between the coach and client. The session lasted approximately 90 minutes and focused on helping the client make a decision about their work schedule and responsibilities for the upcoming academic year. The coach used a combination of questioning, reflection, and a card-based activity to explore the client's preferences and values related to their work.

2. Coaching Context and Contract

The coaching engagement appears to be part of a formal process, possibly related to the client's professional development as an educator. The specific contract was not explicitly discussed in this session, but there were references to previous conversations and agreed-upon goals. The primary focus was on supporting the client in making a decision about their work schedule (0.8 vs 0.9 FTE) and expl

In [63]:
print(report)


Here is a professional coaching evaluation report based on the provided transcript:

Coaching Evaluation Report

1. Session Overview

This was the third and final session in a series of coaching conversations between the coach and client. The session lasted approximately 90 minutes and focused on helping the client make a decision about their work schedule and responsibilities for the upcoming academic year. The coach used a combination of questioning, reflection, and a card-based activity to explore the client's preferences and values related to their work.

2. Coaching Context and Contract

The coaching engagement appears to be part of a formal process, possibly related to the client's professional development as an educator. The specific contract was not explicitly discussed in this session, but there were references to previous conversations and agreed-upon goals. The primary focus was on supporting the client in making a decision about their work schedule (0.8 vs 0.9 FTE) and expl